# Despliegue y promoción

Actualiza Model Serving en Databricks o simula el despliegue localmente.

In [ ]:
import os
import mlflow
from mlflow.tracking import MlflowClient
from iris_mlflow_utils import (
    build_deployment_config,
    build_runtime_config,
    detect_runtime,
    load_dataset_for_runtime,
    promote_champion,
    simulate_local_deployment,
    update_serving_endpoint,
    wait_for_endpoint_ready,
)

runtime_mode = detect_runtime()
config = build_runtime_config(model_slug='random_forest')
deployment_config = build_deployment_config()
if runtime_mode == 'databricks':
    dbutils.widgets.text('model_name', deployment_config.model_name)
    dbutils.widgets.text('model_version', '')
    model_name = dbutils.widgets.get('model_name')
    model_version = dbutils.widgets.get('model_version')
else:
    model_name = deployment_config.model_name
    model_version = os.getenv('IRIS_MODEL_VERSION', '')
if config.tracking_uri:
    mlflow.set_tracking_uri(config.tracking_uri)
mlflow.set_registry_uri(config.registry_uri)
registry = MlflowClient(registry_uri=config.registry_uri)
if not model_version:
    versions = list(registry.search_model_versions(f"name='{model_name}'"))
    if not versions:
        raise RuntimeError(f'No hay versiones para {model_name}.')
    model_version = str(max(versions, key=lambda item: int(item.version)).version)
version = registry.get_model_version(model_name, model_version)
if version.tags.get('evaluation_status') != 'passed':
    raise RuntimeError('La versión no superó evaluación.')
if version.tags.get('approval_status') != 'approved':
    raise RuntimeError('La versión no tiene aprobación.')
if runtime_mode == 'local':
    model = mlflow.pyfunc.load_model(f'models:/{model_name}/{model_version}')
    dataset = load_dataset_for_runtime(runtime_mode='local', spark=None, config=config)
    predictions = model.predict(dataset.features.head(deployment_config.smoke_test_rows))
    smoke_test_passed = len(predictions) == min(deployment_config.smoke_test_rows, len(dataset.features))
    manifest = simulate_local_deployment(
        registry, model_name=model_name, model_version=model_version,
        champion_alias=deployment_config.champion_alias,
        manifest_path=config.deployment_manifest_path, smoke_test_passed=smoke_test_passed,
    )
    print(manifest)
else:
    from databricks.sdk import WorkspaceClient
    workspace = WorkspaceClient()
    update_serving_endpoint(
        workspace, endpoint_name=deployment_config.endpoint_name,
        model_name=model_name, model_version=model_version,
    )
    wait_for_endpoint_ready(
        workspace, deployment_config.endpoint_name,
        timeout_seconds=deployment_config.serving_timeout_seconds,
        poll_seconds=deployment_config.serving_poll_seconds,
    )
    dataset = load_dataset_for_runtime(runtime_mode='databricks', spark=spark, config=config)
    sample = dataset.features.head(deployment_config.smoke_test_rows)
    response = workspace.serving_endpoints.query(
        name=deployment_config.endpoint_name,
        dataframe_split={'columns': [str(column) for column in sample.columns], 'data': sample.values.tolist()},
    )
    predictions = getattr(response, 'predictions', None)
    if predictions is None and isinstance(response, dict):
        predictions = response.get('predictions')
    if not isinstance(predictions, list) or len(predictions) != len(sample):
        raise RuntimeError(f'Smoke test inválido: {response!r}')
    promote_champion(registry, model_name=model_name, model_version=model_version, champion_alias=deployment_config.champion_alias)
    registry.set_model_version_tag(model_name, model_version, 'deployment_status', 'deployed')
    dbutils.jobs.taskValues.set(key='deployment_status', value='deployed')
    print({'runtime': runtime_mode, 'endpoint': deployment_config.endpoint_name, 'model_version': model_version, 'status': 'deployed'})
